In [76]:
### Load packages
import polars as pl

## HROM

In [77]:
### Download HROM metadata
# !wget https://www.decodebiome.org/HROM/data/genome_catalog/HROM_Conspecific-genomes-metadata.tsv

hrom_hq_metadata = (
    pl.read_csv('HROM_Conspecific-genomes-metadata.tsv', separator='\t', 
        columns=['genome_name', 'completeness', 'contamination', 'N50', 'size', 'n_contigs', 'clade_separation_score', 'bioproject', 'sample_name', 'completeness_duplicated_0']
    )
        .rename({'genome_name':'genome', 'N50':'n50', 'n_contigs':'num_contigs', 'size':'genome_length', 'clade_separation_score':'gunc_css', 'bioproject':'project', 'sample_name':'sample', 'completeness_duplicated_0':'species_bin'})
        .filter(
            (pl.col('completeness') >= 90) &
            (pl.col('contamination') <= 5) &
            (pl.col('gunc_css') <= 0.45)
        )
        .with_columns([
            (
                'https://www.decodebiome.org/HROM/data/genome_catalog/HROM_nonredundant_genomes/' +
                pl.col('species_bin') + "/" +
                pl.col('genome') + ".fna"
            ).alias('url')
        ])
)
print(f"Number of HROM HQ genomes: {hrom_hq_metadata.shape[0]}")
hrom_hq_metadata.head(2)

Number of HROM HQ genomes: 72641


genome,completeness,contamination,n50,genome_length,num_contigs,gunc_css,project,sample,species_bin,url
str,f64,f64,i64,i64,i64,f64,str,str,str,str
"""HROM_Genome_0001""",100.0,0.0,45270,2140433,129,0.0,"""PATRIC""","""66851.6(from external database…","""HROM_Genome_0001""","""https://www.decodebiome.org/HR…"
"""HROM_Genome_0002_1""",92.74,2.016,46460,1379462,43,0.21,"""PRJEB31185""","""ERR3307049""","""HROM_Genome_0002""","""https://www.decodebiome.org/HR…"


In [78]:
### Assign HROM HQ genomes to a taxonomy, isolate status, and Q-score
# !wget https://www.decodebiome.org/HROM/data/genome_catalog/HROM-Species-metadata.tsv
hrom_taxonomy = (
    pl.read_csv('HROM-Species-metadata.tsv', separator='\t', columns=['species', 'gtdb_taxonomy', 'genus'])
        .with_columns([
            pl.col('genus').str.replace('g__', '')
        ])
        .rename({'species': 'species_bin', 'gtdb_taxonomy':'taxonomy'})
)

hrom_cluster_data = (
    hrom_hq_metadata
        .join(hrom_taxonomy, on='species_bin',  how='left')
        .with_columns([
            pl.when(pl.col('project').is_in(['NCBI-RefSeq', 'PATRIC', 'HOMD_repository', 'eHOMD', 'GenBank']))
                .then(pl.lit('Isolate'))
                .otherwise(pl.lit('MAG'))
                .alias('source_type')
        ])
        [['genome', 'genome_length', 'n50', 'num_contigs', 'completeness', 'contamination', 'gunc_css', 'project', 'sample', 'taxonomy', 'genus', 'source_type', 'url']]
)
hrom_cluster_data.head(2)

genome,genome_length,n50,num_contigs,completeness,contamination,gunc_css,project,sample,taxonomy,genus,source_type,url
str,i64,i64,i64,f64,f64,f64,str,str,str,str,str,str
"""HROM_Genome_0001""",2140433,45270,129,100.0,0.0,0.0,"""PATRIC""","""66851.6(from external database…","""d__Archaea;p__Methanobacteriot…","""Methanobrevibacter_A""","""Isolate""","""https://www.decodebiome.org/HR…"
"""HROM_Genome_0002_1""",1379462,46460,43,92.74,2.016,0.21,"""PRJEB31185""","""ERR3307049""","""d__Archaea;p__Thermoplasmatota…","""Methanomethylophilus""","""MAG""","""https://www.decodebiome.org/HR…"


## HRGM2

In [79]:
### Download HRGM2 metadata
# !wget https://www.decodebiome.org/HRGM2/data/genome_catalog/Dereplication_genomes_metadata.tsv -O Dereplication_genomes_metadata.tsv

# All HRGM2 genomes are HQ
hrgm2_metadata = (
    pl.read_csv('Dereplication_genomes_metadata.tsv', separator='\t',
        columns=['Genome name', 'Completeness', 'Contamination', 'Data set', 'Sample', 'GTDB Classification', 'Length', 'N50', 'Genome type', 'Non-redundant']
    )
    
)
print(f"Number of HRGM2 HQ genomes: {hrgm2_metadata.shape[0]}")
hrgm2_metadata.head(2)

Number of HRGM2 HQ genomes: 230632


Genome name,Completeness,Contamination,Data set,Sample,GTDB Classification,Length,N50,Genome type,Non-redundant
str,f64,f64,str,str,str,i64,i64,str,bool
"""GENOME000001""",100.0,0.806,"""AWI_Cameroon""","""FBeb1""","""d__Bacteria;p__Actinobacteriot…",2323830,99160,"""MAG""",true
"""GENOME000002""",99.68,0.632,"""AWI_Cameroon""","""FBeb1""","""d__Bacteria;p__Firmicutes_C;c_…",2183119,9644,"""MAG""",true


In [98]:
### Download HRGM2 sample data
# !wget https://static-content.springer.com/esm/art%3A10.1038%2Fs41564-025-02206-1/MediaObjects/41564_2025_2206_MOESM3_ESM.xlsx

hrgm2_hq_metadata = (
    pl.read_excel('41564_2025_2206_MOESM3_ESM.xlsx', sheet_name='S3', drop_empty_rows=True)
        .filter(pl.col('__UNNAMED__1') != 'Original name')
        .rename(
            {'Supplementary Table 3. Metadata for 230,632 redundant NC genomes.':'genome', '__UNNAMED__2': 'completeness',
            '__UNNAMED__3': 'contamination', '__UNNAMED__4': 'project', '__UNNAMED__5': 'sample', '__UNNAMED__6': 'taxonomy', '__UNNAMED__12': 'Genus',
            '__UNNAMED__14': 'genome_length', '__UNNAMED__15': 'n50', '__UNNAMED__16': 'source_type', '__UNNAMED__20': 'non_redundant',
            '__UNNAMED__21': 'num_contigs'}
        )
        .with_columns([
            pl.lit(0.45).alias('gunc_css'),
            pl.col('Genus').str.replace(r'^g__', '').alias('genus'),
            pl.col('genome_length').cast(pl.Int64),
            pl.col('n50').cast(pl.Int64),
            pl.col('completeness').cast(pl.Float64),
            pl.col('contamination').cast(pl.Float64),
            pl.col('num_contigs').cast(pl.Int64)
        ])
        .filter(pl.col('non_redundant') == 'true')
        [['genome', 'genome_length', 'n50', 'num_contigs', 'completeness', 'contamination', 'gunc_css', 'project', 'sample', 'taxonomy', 'genus', 'source_type']]
)

In [99]:
### Download HRGM2 genome URLs
# !wget https://www.decodebiome.org/HRGM2/data/genome_catalog/Total_Genomes/download_link_info.tsv

urls = (
    pl.read_csv('download_link_info.tsv', separator='\t', columns=['Genome name', 'Download link'])
        .rename({'Genome name': 'genome', 'Download link': 'url'})
)

hrgm2_cluster_data = (
    hrgm2_hq_metadata
        .join(urls, on='genome', how='left')
)
hrgm2_cluster_data.head(2)

genome,genome_length,n50,num_contigs,completeness,contamination,gunc_css,project,sample,taxonomy,genus,source_type,url
str,i64,i64,i64,f64,f64,f64,str,str,str,str,str,str
"""GENOME000001""",2323830,99160,38,100.0,0.806,0.45,"""PRJEB30834""","""ERR3097290,ERR3097291""","""d__Bacteria;p__Actinobacteriot…","""Collinsella""","""MAG""","""https://www.decodebiome.org/HR…"
"""GENOME000002""",2183119,9644,326,99.68,0.632,0.45,"""PRJEB30834""","""ERR3097290,ERR3097291""","""d__Bacteria;p__Firmicutes_C;c_…","""Dialister""","""MAG""","""https://www.decodebiome.org/HR…"


## mOTUs-db human MAGs

In [82]:
### Download mOTUs-DB isolates (no already in HRGM2 or HROM)
# !wget https://zenodo.org/records/13325008/files/Supplementary_Table_1.tsv.gz?download=1 -O Supplementary_Table_1.tsv.gz
# !wget https://zenodo.org/records/13325008/files/Supplementary_Table_3.tsv.gz?download=1 -O Supplementary_Table_3.tsv.gz
# !wget https://zenodo.org/records/13325008/files/Supplementary_Table_4.tsv.gz?download=1 -O Supplementary_Table_4.tsv.gz
# downloaded mOTUs-DB metadata by clicking "Export Metadata > All" all here: https://motus-db.org/genome-cols

# load metadata
motus_hq_metadata = (
    pl.read_csv('mOTUsv4.0_genome_summary.tsv', separator='\t',
        columns=['genome','completeness', 'contamination', 'gunc_css', 'domain', 'phylum', 'class_', 'order', 'family', 'genus', 'species', 'n50', 'no_scaffolds', 'genome_size', 'study', 'mag', 'location']
    )
    .filter(
        (pl.col('completeness') >= 90) &
        (pl.col('contamination') <= 5) &
        (pl.col('gunc_css') <= 0.45)
    )
    .with_columns([
        (
            'd__' + pl.col('domain') + ';' +
            'p__' + pl.col('phylum') + ';' +
            'c__' + pl.col('class_') + ';' +
            'o__' + pl.col('order') + ';' +
            'f__' + pl.col('family') + ';' +
            'g__' + pl.col('genus') + ';' +
            's__' + pl.col('species')
        ).alias('taxonomy'),
        pl.when(pl.col('mag')).then(pl.lit('MAG')).otherwise(pl.lit('Isolate')).alias('source_type'),
    ])
    .rename({'no_scaffolds': 'num_contigs', 'genome_size': 'genome_length'})
    [['genome', 'genome_length', 'n50', 'num_contigs', 'completeness', 'contamination', 'gunc_css', 'taxonomy', 'genus', 'source_type', 'study', 'location']]
)
print(f"Number of mOTUs-DB HQ genomes: {motus_hq_metadata.shape[0]}")
motus_hq_metadata.head(2)

Number of mOTUs-DB HQ genomes: 2140568


genome,genome_length,n50,num_contigs,completeness,contamination,gunc_css,taxonomy,genus,source_type,study,location
str,i64,i64,i64,f64,f64,f64,str,str,str,str,str
"""ACIN21-1_SAMN05421555_MAG_0000…",4295120,8978,610,94.2,1.96,0.0,"""d__Bacteria;p__Pseudomonadota;…","""Marinovum""","""MAG""","""ACIN21-1""","""ACIN21-1/ACIN21-1_SAMN05421555…"
"""ACIN21-1_SAMN05421555_MAG_0000…",5180459,140035,67,99.47,1.02,0.18,"""d__Bacteria;p__Actinomycetota;…","""Rhodococcus""","""MAG""","""ACIN21-1""","""ACIN21-1/ACIN21-1_SAMN05421555…"


In [83]:
### Identify HQ MAGs from human-associated metagenomic samples
motus_genome2sample= pl.read_csv('Supplementary_Table_1.tsv.gz', separator='\t', comment_prefix="!", columns=['#GENOME', 'METAGENOMIC_SAMPLE'])
motus_genome2sample.head(2)

#GENOME,METAGENOMIC_SAMPLE
str,str
"""ACIN21-1_SAMN05421555_MAG_0000…","""ACIN21-1_SAMN05421555_METAG"""
"""ACIN21-1_SAMN05421555_MAG_0000…","""ACIN21-1_SAMN05421555_METAG"""


In [84]:
### Identify human-associated metagenomic samples
motus_sample2environment = pl.read_csv('Supplementary_Table_4.tsv.gz', separator='\t', comment_prefix="!", columns=['#SAMPLE', 'BIOSAMPLE', 'STUDY', 'ENVIRONMENT'])
motus_sample2environment.head(2)

#SAMPLE,BIOSAMPLE,STUDY,ENVIRONMENT
str,str,str,str
"""ACIN21-1_SAMN05421555_METAG""","""SAMN05421555""","""ACIN21-1""","""marine metagenome, seawater me…"
"""ACIN21-1_SAMN05421561_METAG""","""SAMN05421561""","""ACIN21-1""","""marine metagenome, seawater me…"


In [85]:
### Identify bioproject of origin
motus_study2bioproject = pl.read_csv('Supplementary_Table_3.tsv.gz', separator='\t', comment_prefix="!", columns=['STUDY ID', 'BIOPROJECT ID'])
motus_study2bioproject.head(2)

STUDY ID,BIOPROJECT ID
str,str
"""ACIN21-1""","""PRJEB44456"""
"""AGAR17-1""","""PRJNA320446"""


In [86]:
motus_mag_cluster_data = (
    motus_hq_metadata
        .join(motus_genome2sample, left_on='genome', right_on='#GENOME', how='inner')
        .join(motus_sample2environment, left_on='METAGENOMIC_SAMPLE', right_on='#SAMPLE', how='left')
        .join(motus_study2bioproject, left_on='STUDY', right_on='STUDY ID', how='left')
        .rename({'BIOSAMPLE': 'sample', 'BIOPROJECT ID': 'project'})
        .with_columns([
            ("https://sunagawalab.ethz.ch/share/MOTUS/database/4.0/data/genomes/" +
                pl.col('STUDY') + "/" +
                pl.col('METAGENOMIC_SAMPLE') + "/" +
                pl.col("genome") + "/" +
                pl.col("genome") + ".fa.gz").alias('url')
        ])
        .filter(pl.col('ENVIRONMENT').str.contains('human'))
        .filter(pl.col('source_type') == 'MAG')
        [['genome', 'genome_length', 'n50', 'num_contigs', 'completeness', 'contamination', 'gunc_css', 'project', 'sample', 'taxonomy', 'genus', 'source_type', 'url']]
)
motus_mag_cluster_data.head(2)

genome,genome_length,n50,num_contigs,completeness,contamination,gunc_css,project,sample,taxonomy,genus,source_type,url
str,i64,i64,i64,f64,f64,f64,str,str,str,str,str,str
"""AOGA20-1_SAMN13565103_MAG_0000…",2795570,28500,151,93.59,0.59,0.0,"""PRJNA595703""","""SAMN13565103""","""d__Bacteria;p__Bacteroidota;c_…","""Prevotella""","""MAG""","""https://sunagawalab.ethz.ch/sh…"
"""AOGA20-1_SAMN13565103_MAG_0000…",1144474,14019,136,95.77,0.0,0.0,"""PRJNA595703""","""SAMN13565103""","""d__Bacteria;p__Actinomycetota;…","""Lancefieldella""","""MAG""","""https://sunagawalab.ethz.ch/sh…"


## mOTUs-db isolates from human-associated taxa

In [87]:
### Identify genera containing HQ human MAGs in mOTUs-DB
print("Number of genera with HQ human MAGs: ", motus_mag_cluster_data.select(pl.col('genus').n_unique()).item())

human_genera = set(motus_mag_cluster_data['genus'])

# human_isolate_genera = 
motus_isolate_cluster_data = (
    motus_hq_metadata
        .filter(pl.col('source_type') == 'Isolate')
        .filter(pl.col('genus').is_in(human_genera))
        .with_columns([
            pl.when(pl.col('genome').str.starts_with('RSGB23'))
                .then(pl.col('genome').str.replace(r'RSGB23-\d_', '').str.replace(r'-V\d_.*', '').str.replace('-', '_'))
                .otherwise(pl.col('genome').str.replace(r'JGIG23-\d_', '').str.replace(r'_GENO_.*', '').str.replace('-', '_'))
            .alias('sample'),
            pl.when(pl.col('study') == 'JGIG23-1').then(pl.lit('JGI')).otherwise(pl.lit('NCBI')).alias('project'),
            (
                "https://sunagawalab.ethz.ch/share/MOTUS/database/4.0/data/genomes/" +
                pl.col('location')
            ).alias('url')
        ])
        [['genome', 'genome_length', 'n50', 'num_contigs', 'completeness', 'contamination', 'gunc_css', 'project', 'sample', 'taxonomy', 'genus', 'source_type', 'url']]
        # .join(motus_study2bioproject, left_on='STUDY', right_on='STUDY ID', how='full', coalesce=True)
)
motus_isolate_cluster_data.head(2)

Number of genera with HQ human MAGs:  1706


genome,genome_length,n50,num_contigs,completeness,contamination,gunc_css,project,sample,taxonomy,genus,source_type,url
str,i64,i64,i64,f64,f64,f64,str,str,str,str,str,str
"""JGIG23-1_GA0056940_GENO_100000…",1895239,1617820,4,100.0,0.0,0.0,"""JGI""","""GA0056940""","""d__Bacteria;p__Actinomycetota;…","""Bifidobacterium""","""Isolate""","""https://sunagawalab.ethz.ch/sh…"
"""JGIG23-1_GA0057557_GENO_100000…",2052470,579125,5,100.0,0.0,0.0,"""JGI""","""GA0057557""","""d__Bacteria;p__Actinomycetota;…","""Bifidobacterium""","""Isolate""","""https://sunagawalab.ethz.ch/sh…"


In [88]:
### Identify GenBank/RefSeq IDs in mOTUs-DB
motus_ncbi_acc = set(
    motus_hq_metadata
        .filter(pl.col('genome').str.starts_with('RSGB23'))
        .with_columns([
            pl.col('genome').str.replace(r'RSGB23-\d_', '').str.replace(r'-V\d_.*', '').str.replace('-', '_').alias('ncbi_accession')
        ])
        ['ncbi_accession']
)

## ProGenomes4

In [89]:
### Download ProGenomes4 metadata
# !wget https://progenomes.embl.de/data/pg4_ncbi_taxonomy.tsv.gz
# !wget https://progenomes.embl.de/data/pg4_consensus_gtdb_taxonomy_per_ani_cluster.tsv.gz
# !wget https://progenomes.embl.de/data/pg4_ANI_clustering.tsv.gz

### Identify ProGenomes genomes in human-associated genera
pg4_human_clusters = set(
    pl.read_csv('pg4_consensus_gtdb_taxonomy_per_ani_cluster.tsv.gz', separator='\t', columns=['ANI_pg4_cluster_id', 'short_assignment'])
        .with_columns([
            pl.col('short_assignment').str.split(' ').list[0].alias('genus')
        ])
        .filter(pl.col('genus').is_in(human_genera))
        ['ANI_pg4_cluster_id']
)

In [90]:
### Identify progenomes genomes in human-associated genera that are not in mOTUs-DB
pg4_clusters = (
    pl.read_csv('pg4_ANI_clustering.tsv.gz', separator='\t', has_header=False, new_columns=['cluster_id', 'members'])
        .with_columns([
            pl.col('members').str.split(';')
        ])
        .explode('members')
        .filter(pl.col('cluster_id').is_in(set(pg4_human_clusters)))
        .with_columns([
            pl.col('members').str.split('.').list[0].alias('members1')
        ])
        .with_columns([
            pl.col('members').str.replace(r'^GCA', 'GCF').alias('members2')
        ])
        .filter(
            (~pl.col('members1').is_in(motus_ncbi_acc)) &
            (~pl.col('members2').is_in(motus_ncbi_acc))
        )
)
print("Number of ProGenomes4 genomes in human-associated genera that are not in mOTUs-DB: ", pg4_clusters.height)

### Progenomes4 non-reps can't be downloaded currently

Number of ProGenomes4 genomes in human-associated genera that are not in mOTUs-DB:  1248145


## Combining all genomes

In [96]:
print(hrom_cluster_data.head(1))
print(hrgm2_cluster_data.head(1))
print(motus_isolate_cluster_data.head(1))
print(motus_mag_cluster_data.head(1))

shape: (1, 13)
┌────────────┬────────────┬───────┬────────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ genome     ┆ genome_len ┆ n50   ┆ num_contig ┆ … ┆ taxonomy  ┆ genus     ┆ source_ty ┆ url       │
│ ---        ┆ gth        ┆ ---   ┆ s          ┆   ┆ ---       ┆ ---       ┆ pe        ┆ ---       │
│ str        ┆ ---        ┆ i64   ┆ ---        ┆   ┆ str       ┆ str       ┆ ---       ┆ str       │
│            ┆ i64        ┆       ┆ i64        ┆   ┆           ┆           ┆ str       ┆           │
╞════════════╪════════════╪═══════╪════════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ HROM_Genom ┆ 2140433    ┆ 45270 ┆ 129        ┆ … ┆ d__Archae ┆ Methanobr ┆ Isolate   ┆ https://w │
│ e_0001     ┆            ┆       ┆            ┆   ┆ a;p__Meth ┆ evibacter ┆           ┆ ww.decode │
│            ┆            ┆       ┆            ┆   ┆ anobacter ┆ _A        ┆           ┆ biome.org │
│            ┆            ┆       ┆            ┆   ┆ iot…      ┆           ┆

In [102]:
### Concatenate all dfs
final_cluster_data = pl.concat([
    hrom_cluster_data,
    hrgm2_cluster_data,
    motus_mag_cluster_data,
    motus_isolate_cluster_data
], how='vertical')
print("Number of rows in final_cluster_data:", final_cluster_data.height)
final_cluster_data.write_csv('final_cluster_data.tsv', separator='\t')

Number of rows in final_cluster_data: 2054956
